In [1]:
import os
import json
import time
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

TOOL_FILE_DIR = Path("./tool_file")
TOOL_FILE_DIR.mkdir(exist_ok=True)

In [2]:
def dump(tool_name: str, data, run_id: str) -> str:
    """모든 툴의 결과는 이 함수를 통해서만 파일로 떨군다. 반환값은 파일 경로 하나.
    에이전트는 이 경로를 읽어서 검증하고, 다른 툴에 넘길 때도 데이터가 아니라 이 경로를 넘긴다."""
    path = TOOL_FILE_DIR / f"{run_id}_{tool_name}.json"
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(path)

In [3]:
def _kakao_keyword(query: str, page: int = 1, category_code: str = "") -> tuple[list[dict], bool]:
    """카카오 키워드 검색 1페이지. (결과 리스트, is_end) 반환."""
    params = {"query": query, "size": 15, "page": page}
    if category_code:
        params["category_group_code"] = category_code

    resp = requests.get(
        "https://dapi.kakao.com/v2/local/search/keyword.json",
        headers={"Authorization": f"KakaoAK {os.environ['KAKAO_REST_API_KEY']}"},
        params=params,
        timeout=5,
    )
    resp.raise_for_status()
    body = resp.json()

    places = [
        {
            "id": d["id"],
            "name": d["place_name"],
            "category": d["category_name"],
            "address": d["road_address_name"] or d["address_name"],
            "url": d["place_url"],
            "source": "kakao_local",
        }
        for d in body["documents"]
    ]
    return places, body["meta"]["is_end"]

In [4]:
_kakao_keyword("창원 귀산 카페")

([{'id': '1114671449',
   'name': '하우요커피',
   'category': '음식점 > 카페',
   'address': '경남 창원시 성산구 삼귀로 524-6',
   'url': 'http://place.map.kakao.com/1114671449',
   'source': 'kakao_local'},
  {'id': '1181055498',
   'name': '엘리브',
   'category': '음식점 > 카페',
   'address': '경남 창원시 성산구 귀산로 19',
   'url': 'http://place.map.kakao.com/1181055498',
   'source': 'kakao_local'},
  {'id': '1846671408',
   'name': '갱고반지하',
   'category': '음식점 > 카페 > 커피전문점',
   'address': '경남 창원시 성산구 삼귀로 530',
   'url': 'http://place.map.kakao.com/1846671408',
   'source': 'kakao_local'},
  {'id': '1570418656',
   'name': '카페모조',
   'category': '음식점 > 카페',
   'address': '경남 창원시 성산구 삼귀로486번길 60',
   'url': 'http://place.map.kakao.com/1570418656',
   'source': 'kakao_local'},
  {'id': '1529832108',
   'name': '커피프렌즈',
   'category': '음식점 > 카페',
   'address': '경남 창원시 성산구 삼귀로 414-23',
   'url': 'http://place.map.kakao.com/1529832108',
   'source': 'kakao_local'},
  {'id': '711813754',
   'name': '카페더원',
   'category': '

In [5]:
import re
import html

def _strip_tags(text: str) -> str:
    """네이버 응답의 <b> 하이라이트 태그와 HTML 엔티티를 제거한다."""
    return html.unescape(re.sub(r"<[^>]+>", "", text)).strip()


def _naver_blog(query: str, display: int = 30, sort: str = "sim") -> list[dict]:
    """네이버 블로그 검색 1회 호출.

    display: 1~100 (기본 10)
    sort: 'sim'(정확도순) 또는 'date'(최신순)
    """
    resp = requests.get(
        "https://openapi.naver.com/v1/search/blog",
        headers={
            "X-Naver-Client-Id": os.environ["NAVER_CLIENT_ID"],
            "X-Naver-Client-Secret": os.environ["NAVER_CLIENT_SECRET"],
        },
        params={"query": query, "display": display, "sort": sort},
        timeout=5,
    )
    resp.raise_for_status()

    return [
        {
            "title": _strip_tags(item["title"]),
            "description": _strip_tags(item["description"]),
            "link": item["link"],
            "postdate": item.get("postdate", ""),
        }
        for item in resp.json()["items"]
    ]

In [6]:
_naver_blog("창원 가포 오션뷰 카페")

[{'title': '[창원카페추천] 페리데스 반전매력 오션뷰 베이커리 맛집 내돈내산 ....',
  'description': '[창원카페추천] 페리데스 오션뷰 베이커리 맛집 내돈내산 후기 안녕하세요... 다녀와보니, 창원에서 뷰 좋은 베이커리 카페 찾으신다면 페리데스 진짜 추천드려요  가포 앞바다 보면서 갓 구운 빵이랑 커피 한 잔...',
  'link': 'https://blog.naver.com/canada179/224369721321',
  'postdate': '20260806'},
 {'title': '마산 가포 오션뷰 카페 페리데스, 커피보다 분위기에 먼저 반했던 ....',
  'description': '예쁜 곳보다 오래 머물고 싶은 공간을 찾는다면 페리데스를 추천드립니다. #마산카페 #가포카페 #페리데스 #마산오션뷰카페 #마산베이커리카페 #창원카페추천 #가포드라이브 #마산데이트 #경남카페 #대형카페',
  'link': 'https://blog.naver.com/dwd0684/224336449681',
  'postdate': '20260704'},
 {'title': '바다보며 힐링하기 좋은 마산 가포 오션뷰 신상카페 기운상자',
  'description': '바다보며 힐링하기 좋은 마산 가포 오션뷰 신상카페 기운상자 창원 마산 가포에 새롭게 문을 연 신상카페가 있다고 해서 달려가 보았습니다. 바다를 품은 오션뷰 카페가 많아지고 있는 요즘, 또 하나의...',
  'link': 'https://blog.naver.com/pigdanny0113/224309168114',
  'postdate': '20260608'},
 {'title': '(창원) 마산 가포 오션뷰카페 지중해 후기',
  'description': 'ㅋㅋ 주차장도 넓어서 좋고 진짜 바다 바로 코앞인데다가 제주도 느낌까지 느낄 수 있는 카페라서 너무 좋았어요 가족 나들이, 데이트코스로도 너무 좋을것같아요 창원,마산 가포 오션뷰카페 지중해 추천드려요 !',
  'link':

In [7]:
from google import genai
from google.genai import types

_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
UTIL_MODEL = "gemini-3.1-flash-lite"

def _llm(prompt: str, *, json_mode: bool = False, temperature: float = 0.0) -> str:
    """유틸 LLM 단발 호출. stateless, 텍스트만 반환."""
    config = types.GenerateContentConfig(temperature=temperature)
    if json_mode:
        config.response_mime_type = "application/json"

    resp = _client.models.generate_content(
        model=UTIL_MODEL,
        contents=prompt,
        config=config,
    )
    return resp.text

In [ ]:
import json

_EXTRACT_PLACE_NAMES_SYSTEM_PROMPT = """\
아래는 네이버 블로그 검색 결과다. 각 줄은 '제목 | 요약' 형식이다.
여기서 실제 존재하는 가게/장소의 고유 상호명만 뽑아라.
- 제목에 상호명이 있을 확률이 높다. 요약은 보조로만 참고한다.
- '카페 추천', '맛집 베스트10', '데이트코스' 같은 일반명사구는 상호명이 아니다. 
- 같은 매장이 여러 줄에 걸쳐 나오면 한 번만 적어라.
- 확실하지 않으면 넣지 마라.
- 결과는 문자열 배열 JSON으로만 출력한다. 예: ["블루보틀 성수", "어니언 성수"]
"""

def _extract_place_names(blog_items: list[dict]) -> list[str]:
    """네이버 블로그 검색 결과 뭉치에서 실제 매장/장소 상호명만 추출한다.
    title에 상호명이 있을 확률이 높고 description은 보조 참고용이다.
    반환값은 검증 전 후보 목록이며, 호출부에서 카카오 역조회로 실존을 확인해야 한다."""
    if not blog_items:
        return []

    lines = "\n".join(
        f"{i+1}. {it['title']} | {it['description']}"
        for i, it in enumerate(blog_items)
    )
    prompt = f"{_EXTRACT_PLACE_NAMES_SYSTEM_PROMPT}\n{lines}"

    raw = _llm(prompt, json_mode=True)
    try:
        names = json.loads(raw)
    except json.JSONDecodeError:
        return []

    if not isinstance(names, list):
        return []
    return [n.strip() for n in names if isinstance(n, str) and n.strip()]

In [39]:
from google.adk.tools import ToolContext


def collect_places(
    query: str,
    tool_context: ToolContext,
    # target_count: int = 20,
    category_code: str = "",
) -> dict:
    """지역 키워드로 매장 후보 목록을 수집한다.

    카카오 로컬 검색을 우선 사용하며, 결과가 부족하면 재시도를 요청한다.
    3회까지 부족하면 네이버 블로그에서 매장명을 추출해 카카오로 검증한 뒤 보충한다.

    검색어는 '지역 + 짧은 업종어' 형태여야 한다. 서술형 문장은 결과가 거의 나오지 않는다.
      좋은 예: '성수동 카페', '성수동 카공카페'
      나쁜 예: '성수동에서 조용히 작업하기 좋은 카페'

    Args:
        query: 검색어. '지역 + 짧은 업종어' 조합.
        target_count: 확보하려는 결과 개수. 최대 45까지만 가능하다.
        category_code: 카테고리 필터. 'CE7'=카페, 'FD6'=음식점, 'AT4'=관광명소, 'CT1'=문화시설.
                       불필요하면 빈 문자열로 둔다.

    Returns:
        status가 'retry'면 결과가 부족한 것이므로, message의 안내에 따라
        query를 더 짧고 단순하게 바꿔서 이 툴을 다시 호출해야 한다.
        status가 'ok'면 수집이 끝난 것이므로 다시 호출하지 말고 file 경로를 사용한다.
        예시(재시도): {'status': 'retry', 'message': '결과가 부족합니다...'}
        예시(완료): {'status': 'ok', 'count': 20, 'file': './tool_file/collect_places_123.json'}
    """
    attempt_key = "temp:collect_attempts"
    attempts = tool_context.state.get(attempt_key, 0) + 1
    tool_context.state[attempt_key] = attempts

    places: dict[str, dict] = {}
    target_count: int = 40

    # 1) 카카오 로컬 페이지네이션
    page = 1
    while len(places) < target_count and page <= 45:
        batch, is_end = _kakao_keyword(query, page=page, category_code=category_code)
        for p in batch:
            places.setdefault(p["id"], p)
        if is_end:
            break
        page += 1

    # 2) 부족한데 아직 재시도 여유가 있으면 → 에이전트에게 쿼리 재구성 요청
    if len(places) < target_count * 0.5 and attempts < 3:
        return {
            "status": "retry",
            "message": (
                f"'{query}' 검색 결과가 {len(places)}개로 부족합니다. ({attempts}/3번째 시도) "
                "핵심 지역명은 유지하고 수식어를 제거해 더 짧은 업종어로 바꿔 다시 호출하세요. "
                "예: '조용히 작업하기 좋은 카페' → '카공카페' 또는 '스터디카페'"
            ),
        }

    # 3) 3회까지 부족하면 → 네이버 블로그에서 매장명 추출 후 카카오로 검증
    if len(places) < target_count * 0.5:
        blog_items = _naver_blog(query, display=30, sort="sim")
        print("blog_items: ", blog_items)
        for name in _extract_place_names(blog_items):
            if len(places) >= target_count:
                break
            verified, _ = _kakao_keyword(name, page=1)
            print("name:", verified)
            for p in verified:
                if p["id"] not in places:
                    p["source"] = "blog_derived"
                    places[p["id"]] = p
                    break

    result = list(places.values())[:target_count]
    file_path = dump("collect_places", result, _run_id(tool_context))
    tool_context.state["temp:last_collect_places_file"] = file_path

    return {
        "status": "ok",
        "count": len(result),
        "source_breakdown": {
            "local": sum(1 for p in result if p.get("source") == "kakao_local"),
            "blog_derived": sum(1 for p in result if p.get("source") == "blog_derived"),
        },
        "file": file_path,
    }

In [10]:
def _run_id(tool_context: ToolContext) -> str:
    """현재 invocation의 식별자. 없으면 최초 호출 시점으로 생성한다."""
    key = "temp:run_id"
    rid = tool_context.state.get(key)
    if rid is None:
        rid = str(int(time.time() * 1000))
        tool_context.state[key] = rid
    return rid

In [11]:
# class FakeToolContext:
#     def __init__(self, invocation_id="test-run"):
#         self.state = {}
#         self.invocation_id = invocation_id

# ctx = FakeToolContext()


# ctx = FakeToolContext()
# r1 = collect_places("신림동 서울 밥집", ctx)  # attempts=1
# print(r1)
# r2 = collect_places("신림동 카페", ctx)          # attempts=2
# print(r2)
# r3 = collect_places("신림동 카페", ctx)               # attempts=3 → 네이버 폴백
# print(r3)

In [12]:
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types

APP_NAME = "wh_poc"
USER_ID = "user1"
SESSION_ID = "s1"

INSTRUCTION = """당신은 지역 장소를 찾아주는 어시스턴트입니다.

사용자가 장소를 찾아달라고 하면:
1. 사용자의 자연어 요청에서 '지역 + 짧은 업종어' 형태의 검색어를 만드세요.
   예: "성수동에서 조용히 작업할 카페" → "성수동 카공카페"
2. collect_places 툴을 호출하세요.
3. 반환된 status를 반드시 확인하세요.

status가 'retry'인 경우:
- message의 안내대로 query를 더 짧고 단순하게 바꿔서 collect_places를 즉시 다시 호출하세요.
- 사용자에게 되묻지 말고, 스스로 검색어를 바꿔서 재시도하세요.

status가 'ok'인 경우:
- 더 이상 collect_places를 호출하지 마세요.
- count와 file 경로를 사용자에게 알려주고 결과를 정리해 답변하세요.
"""

agent = LlmAgent(
    model="gemini-3.1-flash-lite",
    name="place_finder",
    description="지역 기반 장소 추천 에이전트",
    instruction=INSTRUCTION,
    tools=[collect_places],
)

runner = InMemoryRunner(agent=agent, app_name=APP_NAME)

In [13]:
import uuid

async def run(query: str):
    session_id = str(uuid.uuid4())
    await runner.session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    content = types.Content(role="user", parts=[types.Part(text=query)])

    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.function_call:
                    print(f"[TOOL CALL] {part.function_call.name}({dict(part.function_call.args)})")
                elif part.function_response:
                    print(f"[TOOL RESULT] {part.function_response.response}")
                elif part.text and part.text.strip():
                    print(f"[TEXT] {part.text.strip()}")

await run("신림역 혼밥 하기 좋은 백반집 추천좀")

d:\python_project\welcom_to_hometown\.venv\Lib\site-packages\google\adk\tools\function_tool.py:96: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


[TOOL CALL] collect_places({'query': '신림역 백반집', 'category_code': 'FD6'})
[TOOL RESULT] {'status': 'ok', 'count': 40, 'source_breakdown': {'local': 40, 'blog_derived': 0}, 'file': 'tool_file\\1786583727519_collect_places.json'}
[TEXT] 신림역 근처에서 혼밥하기 좋은 백반집을 40곳 찾았습니다. 편안하게 식사하실 수 있는 몇 곳을 추천해 드립니다.

**추천 백반집 리스트:**

1. **전주집**: 깔끔하고 정갈한 가정식 백반으로 인기가 많으며, 혼밥하기에도 부담 없는 분위기입니다.
2. **시골밥상**: 집밥 같은 푸근한 느낌의 백반을 제공하며, 가격대도 합리적이라 혼자 식사하시는 분들이 자주 찾는 곳입니다.
3. **남도식당**: 든든한 한 끼를 원하실 때 방문하기 좋으며, 다양한 반찬이 정성스럽게 나옵니다.

더 자세한 목록이나 특정 스타일(가격대, 거리 등)을 원하시면 말씀해 주세요! 위 장소들에 대한 상세 정보는 아래 파일에서 확인하실 수 있습니다.

파일 경로: `./tool_file/1786583727519_collect_places.json`


In [41]:
import re
import json
import asyncio
import httpx
from collections import Counter


def _load_places(file_path: str) -> list[dict]:
    with open(file_path, encoding="utf-8") as f:
        return json.load(f)


NAVER_SEM = asyncio.Semaphore(10)

async def _collect_info_urls(places: list[dict], fetch_display: int = 10, target_per_place: int = 5) -> dict[str, list[dict]]:
    async def _naver_blog(query: str) -> tuple[list[dict], str]:
        async with NAVER_SEM:
            for attempt in range(3):
                async with httpx.AsyncClient(timeout=5) as client:
                    resp = await client.get(
                        "https://openapi.naver.com/v1/search/blog",
                        headers={
                            "X-Naver-Client-Id": os.environ["NAVER_CLIENT_ID"],
                            "X-Naver-Client-Secret": os.environ["NAVER_CLIENT_SECRET"],
                        },
                        params={"query": query, "display": fetch_display, "sort": "date"},
                    )
                if resp.status_code == 429:
                    print(f"[429] '{query}' (attempt {attempt + 1}/3)")
                    await asyncio.sleep(0.5 * (attempt + 1))
                    continue
                resp.raise_for_status()
                break
            else:
                return [], "429_exhausted"

            items = resp.json()["items"]
            if not items:
                return [], "empty_api_result"

            return [
                {
                    "title": _strip_tags(item["title"]),
                    "description": _strip_tags(item["description"]),
                    "link": item["link"],
                    "postdate": item.get("postdate", ""),
                }
                for item in items
            ], "ok"

    queries = []
    for p in places:
        road = ""
        for token in p.get("address", "").split():
            m = re.search(r"(로|길)", token)
            if m:
                road = token[:m.start()]
                break
        queries.append(f"{road} {p['name']}".strip())

    raw = await asyncio.gather(*[_naver_blog(q) for q in queries])

    result = {}
    reasons = {}
    for p, (items, status) in zip(places, raw):
        filtered = [
            b for b in items
            if p["name"] in b["title"] or p["name"] in b["description"]
        ][:target_per_place]
        result[p["name"]] = filtered
        if not filtered:
            reasons[p["name"]] = status if status != "ok" else "no_match_after_filter"

    print("\n실패 사유별 집계:", Counter(reasons.values()))
    print(reasons)

    return result

In [16]:
import time
places = _load_places("./tool_file/1786583727519_collect_places.json")

start = time.time()
urls = await _collect_info_urls(places)
print(f"\n총 소요: {time.time() - start:.1f}초")

[429] '신림동 본도시락 신림점' (attempt 1/3)
[429] '신림 배고픈덮밥이 본점' (attempt 1/3)
[429] '봉천 전주식당' (attempt 1/3)
[429] '남부순환 해탄 봉천역점' (attempt 1/3)
[429] '보라매로5가 장원분식' (attempt 1/3)
[429] '남부순환로163가 보리떡과물고기' (attempt 1/3)
[429] '보라매로2 오소리식당' (attempt 1/3)
[429] '관천로22 밥풀릭스 신림점' (attempt 1/3)
[429] '보라매로3 제주보쌈' (attempt 1/3)
[429] '보라매로5 대박생고기김치찌개' (attempt 1/3)
[429] '보라매로3 제주보쌈' (attempt 2/3)
[429] '당곡 재근국밥' (attempt 1/3)
[429] '보라매로5 강촌민물매운탕 2호점' (attempt 1/3)
[429] '보라매로5 옛집' (attempt 1/3)
[429] '보라매 옛날솥뚜껑생삼겹살' (attempt 1/3)
[429] '신림로18가 옛날불고기' (attempt 1/3)
[429] '은천로6 또와포차' (attempt 1/3)
[429] '은천로6 고향의진미' (attempt 1/3)
[429] '보라매로3 명희네분식' (attempt 1/3)
[429] '보라매로5 한솥도시락 보라매삼성점' (attempt 1/3)
[429] '당곡 재근국밥' (attempt 2/3)
[429] '보라매로5 청가맛' (attempt 1/3)
[429] '쑥고개 계절담은밥상' (attempt 1/3)
[429] '남부순환로161가 현주네퓨전분식' (attempt 1/3)
[429] '난곡 돈벼락삼겹살' (attempt 1/3)
[429] '난곡로65 본가숯불갈비' (attempt 1/3)
[429] '남부순환 남덕식당' (attempt 1/3)
[429] '대방천 옥돌백반전문' (attempt 1/3)
[429] '법원단지 행복한식당' (attempt 1/3)
[429

In [17]:
urls

{'배바우': [{'title': '서울 신림 백반맛집 배바우식당',
   'description': '식당 앞 배바우식당(관악구 신림로 56길 7, 02-3285-1102)은 내부도 정갈하고 반찬도 다양하고... 2호선 신림역 2번 출구에서 323m, 신림선 서원역 1번 출구에서 476m 거리이다. 상차림 저녁 10시까지...',
   'link': 'https://blog.naver.com/hangbogie/224117482976',
   'postdate': '20251221'},
  {'title': '신림 2n첩 반상 이모카세 제육과 생선구이 정식 맛집, 배바우식당',
   'description': '폭발 신림 이모카세 맛집입니당 배바우식당 서울 관악구 신림로56길 7 매일 12:00 - 23:00 02-3285-1102 화장실 외부 식사, 반주, 낮술, 밤술 모두 가능 예약 가능 단체 가능 #신림맛집 #신림역맛집 #신림한식...',
   'link': 'https://blog.naver.com/tifhvos/223849090954',
   'postdate': '20250428'}],
 '서평면옥': [{'title': '서울 보라매역 보라매병원역 서평면옥 이북음식 평양냉면 직장인 ....',
   'description': '위치 : 서평면옥(서울 동작구 보라매로5길 15 전문건설회관 지하1층) - (신림선) 보라매병원역 2번 출구에서 26m * ⏰ 영업시간 : 11:00-21:00... (나도 여기가 첫 평냉 입문이었기에..) ⭐ 총평 평점 ★★★★☆ (4.5/5)...',
   'link': 'https://blog.naver.com/with_jeongeun/224355823381',
   'postdate': '20260723'},
  {'title': '보라매 평양냉면 맛집 서평면옥 평양물냉면 비빔냉면 만두 후기 주....',
   'description': '서평면옥 서울 동작구 보라매로5길 15 전문건설회관 지하1층 11:00-21:0

In [18]:
import re
import asyncio
import httpx
from bs4 import BeautifulSoup

SCRAPE_SEM = asyncio.Semaphore(20)
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)
_FAIL_PREFIX = "__SCRAPE_FAIL__"
_BODY_SELECTORS = [
    "div.se-main-container",   # 스마트에디터 ONE (최신)
    "#postViewArea",           # 구버전 에디터
    "div#post-view",
]


async def scrape_urls(urls: list[str]) -> list[dict]:
    """URL 리스트를 병렬로 긁어 [{'url': ..., 'text': ...}] 반환.
    실패 시 text가 __SCRAPE_FAIL__로 시작한다 (본문이 대괄호로 시작하는 경우와
    혼동되지 않도록 일반 텍스트에 등장할 일 없는 접두사 사용)."""

    def resolve_naver_url(url: str) -> str:
        m = re.search(r"blog\.naver\.com/([^/?]+)/(\d+)", url)
        if m:
            return f"https://blog.naver.com/PostView.naver?blogId={m.group(1)}&logNo={m.group(2)}"
        return url

    def extract_body(soup: BeautifulSoup) -> str:
        for tag in soup(["script", "style"]):
            tag.decompose()
        for selector in _BODY_SELECTORS:
            node = soup.select_one(selector)
            if node:
                text = re.sub(r"\s+", " ", node.get_text(separator=" ", strip=True)).strip()
                if len(text) > 150:
                    return text
        return re.sub(r"\s+", " ", (soup.body or soup).get_text(separator=" ", strip=True)).strip()

    async def fetch_one(client: httpx.AsyncClient, url: str) -> str:
        async with SCRAPE_SEM:
            try:
                resp = await client.get(resolve_naver_url(url))
                resp.raise_for_status()
            except httpx.HTTPError as e:
                return f"{_FAIL_PREFIX}열기 실패: {type(e).__name__}"

        text = extract_body(BeautifulSoup(resp.text, "html.parser"))
        return text if len(text) >= 150 else f"{_FAIL_PREFIX}본문 추출 실패"

    async with httpx.AsyncClient(
        headers={"User-Agent": USER_AGENT}, timeout=10.0, follow_redirects=True
    ) as client:
        texts = await asyncio.gather(*(fetch_one(client, u) for u in urls))

    return [{"url": u, "text": t} for u, t in zip(urls, texts)]

In [19]:
async def scrape_all(urls: dict[str, list[dict]]) -> dict[str, list[dict]]:
    """urls: {매장명: [{title, description, link, postdate}, ...]}
    각 매장의 URL 항목에 'text' 필드를 채워서 같은 구조로 반환한다."""
    all_links = []
    owner = []  # (매장명, 그 매장 리스트 내 인덱스)

    for name, items in urls.items():
        for idx, item in enumerate(items):
            all_links.append(item["link"])
            owner.append((name, idx))

    scraped = await scrape_urls(all_links)

    result = {name: [dict(item) for item in items] for name, items in urls.items()}
    for (name, idx), s in zip(owner, scraped):
        result[name][idx]["text"] = s["text"]

    return result

In [20]:
all_scraped = await scrape_all(urls)

real_fail = sum(
    1 for items in all_scraped.values() for item in items
    if item["text"].startswith(_FAIL_PREFIX)
)
print(f"{real_fail}개 실제 실패 (총 {sum(len(v) for v in all_scraped.values())}개 중)")

0개 실제 실패 (총 67개 중)


In [21]:
async def _llm_async(prompt: str, *, json_mode: bool = False, temperature: float = 0.0) -> str:
    """유틸 LLM 비동기 호출. 병렬 처리용."""
    config = types.GenerateContentConfig(temperature=temperature)
    if json_mode:
        config.response_mime_type = "application/json"

    resp = await _client.aio.models.generate_content(
        model=UTIL_MODEL,
        contents=prompt,
        config=config,
    )
    return resp.text

In [22]:
LLM_SEM = asyncio.Semaphore(10)
_SUMMARIZE_SYSTEM_PROMPT = """\
사용자 요청: "{user_query}"

아래는 '{place_name}'에 대한 블로그 후기 {n}건이다.
여러 후기를 종합해 위 요청 관점에서 이 매장을 평가하고 다음 JSON 형식으로만 답해라.

{{
  "strengths": ["요청 조건에 부합하는 점 2~4개"],
  "weaknesses": ["요청 조건에 아쉬운 점 또는 주의할 점 0~3개"],
  "summary": "전반적인 평가 2~3문장"
}}

규칙:
- 여러 후기에서 반복되는 내용일수록 신뢰도가 높다. 한 후기에만 나오는 개인적 감상은 비중을 낮춰라.
- 후기 간 내용이 상충하면 그 사실을 그대로 적어라. (예: "시간대에 따라 웨이팅 편차가 큼")
- 본문에 실제로 언급된 내용만 사용하고 추측하지 마라.
- 홍보성 과장 표현은 걷어내고 실제 경험 위주로 판단해라.
- 후기가 이 매장에 대한 내용이 아니거나 정보가 부실하면
  {{"strengths": [], "weaknesses": [], "summary": "요약 불가"}}로 답해라.
"""


async def summarize_all(scraped: dict[str, list[dict]], user_query: str) -> dict[str, dict]:
    """매장별로 모든 리뷰를 한 번에 넣어 통합 요약을 만든다.
    반환: {매장명: {"strengths": [...], "weaknesses": [...], "summary": ..., "review_count": n}}"""

    async def summarize_one(place_name: str, texts: list[str]) -> dict:
        joined = "\n\n---\n\n".join(
            f"[후기 {i+1}]\n{t[:2000]}" for i, t in enumerate(texts)
        )
        prompt = (
            _SUMMARIZE_SYSTEM_PROMPT.format(
                user_query=user_query, place_name=place_name, n=len(texts)
            )
            + "\n"
            + joined
        )
        async with LLM_SEM:
            raw = await _llm_async(prompt, json_mode=True)
        try:
            result = json.loads(raw)
        except json.JSONDecodeError:
            result = {"strengths": [], "weaknesses": [], "summary": "요약 불가"}
        result["review_count"] = len(texts)
        return result

    names, tasks = [], []
    for name, items in scraped.items():
        texts = [i["text"] for i in items if not i["text"].startswith(_FAIL_PREFIX)]
        if not texts:
            continue
        names.append(name)
        tasks.append(summarize_one(name, texts))

    results = await asyncio.gather(*tasks)
    return dict(zip(names, results))

In [23]:
summarized = await summarize_all(all_scraped, "신림역 혼밥하기 좋은 백반집")

In [24]:
summarized

{'배바우': {'strengths': ['다양하고 정갈한 반찬이 제공되는 집밥 스타일의 한식당',
   '화학조미료를 거의 사용하지 않아 뒷맛이 깔끔함',
   '신림역에서 도보 5분 거리로 접근성이 좋음',
   '늦은 시간까지 영업하여 여유로운 식사가 가능함'],
  'weaknesses': ['식당이 지하에 위치함', '화장실이 외부(매장 밖)에 있음'],
  'summary': '신림역 인근에서 정갈한 집밥을 즐길 수 있는 곳으로, 다양한 반찬과 깔끔한 맛이 특징입니다. 매장이 넓고 늦은 시간까지 운영하여 혼밥은 물론 단체 식사나 반주를 곁들이기에도 적합합니다.',
  'review_count': 2},
 '서평면옥': {'strengths': ['1인석이 마련되어 있어 혼밥하기에 최적화된 환경임',
   '메뉴가 빨리 나오는 편이라 혼자서도 빠르게 식사 가능',
   '제육, 만두, 녹두지짐이 등 사이드 메뉴를 반 접시 단위로 주문할 수 있어 혼자서도 다양하게 즐기기 좋음'],
  'weaknesses': ['점심 피크 시간대에는 웨이팅이 발생할 수 있음', '일부 메뉴(불고기 등)는 2인 이상 주문 가능함'],
  'summary': '신림역 인근(보라매병원역)에서 혼밥하기 매우 좋은 평양냉면 전문점입니다. 1인석이 잘 갖춰져 있고 사이드 메뉴를 소량으로 주문할 수 있어 혼자 방문해도 부담 없이 다양한 음식을 즐길 수 있습니다.',
  'review_count': 5},
 '도화불백': {'strengths': ['혼밥하는 손님이 항상 있을 정도로 혼자 식사하기에 편안한 분위기',
   '불백과 제육 등 1인분 주문이 가능하며 정갈한 반찬과 쌈 채소가 함께 제공됨',
   '단품 주문 시 미역국이 제공되어 국물 요리를 곁들일 수 있음'],
  'weaknesses': ['신림역에서 도보로 약 8분 정도 소요되어 역과는 다소 거리가 있음',
   '주차 공간이 따로 마련되어 있지 않아 인근 공영주차장을 이용해야 함'],
  'summary': '도화

In [25]:
async def collect_info(tool_context: ToolContext, user_query: str) -> dict:
    """collect_places로 찾은 매장들의 블로그 정보를 수집·요약한다.

    각 매장의 블로그 글 URL을 모아 파일로 남기고, 본문을 스크래핑해
    사용자 요청 관점에서 강점·약점 중심으로 요약한 결과를 별도 파일로 남긴다.
    본문 원문은 저장하지 않는다.

    Args:
        user_query: 사용자의 원래 요청. 요약 시 어떤 관점에서 평가할지 결정한다.
                    예: '신림역 혼밥하기 좋은 백반집'

    Returns:
        {"count": int, "no_info": int, "url_file": str, "summary_file": str}
        summary_file에 매장별 강점·약점·종합평가가 들어있다.
    """
    places_file = tool_context.state.get("temp:last_collect_places_file")
    if not places_file:
        return {"status": "error", "message": "collect_places를 먼저 호출해야 합니다."}

    places = _load_places(places_file)
    run_id = _run_id(tool_context)

    urls = await _collect_info_urls(places)
    url_file = dump("info_urls", urls, run_id)

    scraped = await scrape_all(urls)
    summaries = await summarize_all(scraped, user_query)
    summary_file = dump("info_summary", summaries, run_id)

    return {
        "count": len(summaries),
        "no_info": len(places) - len(summaries),
        "url_file": url_file,
        "summary_file": summary_file,
    }

In [26]:
def read_file(file_path: str) -> dict:
    """툴이 생성한 결과 파일의 내용을 읽는다.
    collect_places나 collect_info가 반환한 file 경로를 넘기면 그 내용을 볼 수 있다.
    최종 답변을 만들기 전에 summary_file을 읽어 매장별 평가를 확인하라.

    Args:
        file_path: 읽을 파일 경로. 이전 툴이 반환한 file/summary_file/url_file 값.

    Returns:
        {"content": ...} 파일 내용. 경로가 잘못되었으면 status가 'error'.
    """
    path = Path(file_path)
    if not path.exists():
        return {"status": "error", "message": f"파일을 찾을 수 없습니다: {file_path}"}

    try:
        return {"content": json.loads(path.read_text(encoding="utf-8"))}
    except json.JSONDecodeError:
        return {"status": "error", "message": "파일을 읽을 수 없습니다."}

In [27]:
INSTRUCTION = """당신은 지역 장소를 찾아 추천하는 어시스턴트입니다.

작업 순서:
1. collect_places로 매장 후보를 수집합니다.
   - status가 'retry'면 message 안내대로 query를 짧게 바꿔 즉시 다시 호출하세요.
   - 사용자에게 되묻지 말고 스스로 검색어를 조정하세요.
2. collect_info로 매장별 블로그 정보를 수집·요약합니다.
   - user_query에는 사용자의 원래 요청을 그대로 넘기세요.
3. read_file로 summary_file을 읽어 매장별 강점·약점을 확인합니다.
4. 사용자 요청에 가장 잘 맞는 5~8곳을 골라 추천합니다.
   - 후기 건수(review_count)가 많을수록 신뢰도가 높습니다.
   - 강점이 요청 조건과 맞는지, 약점이 결정적이지 않은지 따져보세요.
   - 각 추천마다 왜 이곳인지 근거를 간단히 붙이세요.
"""

agent = LlmAgent(
    model="gemini-3.1-flash-lite",
    name="place_finder",
    description="지역 기반 장소 추천 에이전트",
    instruction=INSTRUCTION,
    tools=[collect_places, collect_info, read_file],
)

runner = InMemoryRunner(agent=agent, app_name=APP_NAME)

In [28]:
import uuid
from google.genai import types

async def run(query: str):
    session_id = str(uuid.uuid4())
    await runner.session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    content = types.Content(role="user", parts=[types.Part(text=query)])

    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.function_call:
                    print(f"[TOOL CALL] {part.function_call.name}({dict(part.function_call.args)})")
                elif part.function_response:
                    resp = part.function_response.response
                    # 파일 내용은 길어서 요약만 출력
                    if isinstance(resp, dict) and "content" in resp:
                        print(f"[TOOL RESULT] read_file → {len(str(resp['content']))}자")
                    else:
                        print(f"[TOOL RESULT] {resp}")
                elif part.text and part.text.strip():
                    print(f"[TEXT] {part.text.strip()}")

await run("신림역에서 혼밥하기 좋은 백반집 추천해줘")

[TOOL CALL] collect_places({'category_code': 'FD6', 'query': '신림역 백반'})
[TOOL RESULT] {'status': 'ok', 'count': 40, 'source_breakdown': {'local': 40, 'blog_derived': 0}, 'file': 'tool_file\\1786583785494_collect_places.json'}
[TOOL CALL] collect_info({'user_query': '신림역에서 혼밥하기 좋은 백반집 추천해줘'})
[429] '신림동 본도시락 신림점' (attempt 1/3)
[429] '봉천 전주식당' (attempt 1/3)
[429] '보라매로3 제주보쌈' (attempt 1/3)
[429] '보라매로5가 장원분식' (attempt 1/3)
[429] '관천로22 밥풀릭스 신림점' (attempt 1/3)
[429] '신림 배고픈덮밥이 본점' (attempt 1/3)
[429] '남부순환로163가 보리떡과물고기' (attempt 1/3)
[429] '남부순환 해탄 봉천역점' (attempt 1/3)
[429] '보라매로2 오소리식당' (attempt 1/3)
[429] '보라매로5 대박생고기김치찌개' (attempt 1/3)
[429] '당곡 재근국밥' (attempt 1/3)
[429] '당곡 군산식당' (attempt 1/3)
[429] '보라매로5 강촌민물매운탕 2호점' (attempt 1/3)
[429] '보라매 옛날솥뚜껑생삼겹살' (attempt 1/3)
[429] '보라매로5 옛집' (attempt 1/3)
[429] '신림로18가 옛날불고기' (attempt 1/3)
[429] '은천로6 또와포차' (attempt 1/3)
[429] '은천로6 고향의진미' (attempt 1/3)
[429] '보라매로3 명희네분식' (attempt 1/3)
[429] '보라매로5 한솥도시락 보라매삼성점' (attempt 1/3)
[429] '남부순환 동현

In [44]:
from pydantic import BaseModel, Field
from typing import List


class PlaceFeed(BaseModel):
    name: str = Field(description="매장명")
    address: str = Field(description="도로명 주소")
    reason: str = Field(description="추천 이유. 왜 이 요청에 맞는지 2~3문장")
    highlights: List[str] = Field(description="핵심 강점 2~3개, 짧은 구문으로")
    caution: str = Field(default="", description="주의사항. 없으면 빈 문자열")


class FeedOutput(BaseModel):
    places: List[PlaceFeed] = Field(description="추천 매장 5~8곳")


INSTRUCTION = """당신은 지역 장소를 찾아 추천하는 어시스턴트입니다.

작업 순서:
1. collect_places로 매장 후보를 수집합니다.
   - status가 'retry'면 message 안내대로 query를 짧게 바꿔 즉시 다시 호출하세요.
   - 사용자에게 되묻지 말고 스스로 검색어를 조정하세요.
2. collect_info로 매장별 블로그 정보를 수집·요약합니다.
   - user_query에는 사용자의 원래 요청을 그대로 넘기세요.
3. read_file로 summary_file을 읽어 매장별 강점·약점을 확인합니다.
   - 주소가 필요하면 collect_places가 만든 file도 read_file로 읽으세요.
4. 사용자 요청에 가장 잘 맞는 5~8곳을 골라 JSON으로 응답합니다.
   - 후기 건수(review_count)가 많을수록 신뢰도가 높습니다.
   - 강점이 요청 조건과 맞는지, 약점이 결정적이지 않은지 따져보세요.
   - highlights는 짧은 구문으로 쓰세요. (예: "1인석 있음", "반찬 무한리필")
   - 파일에 없는 정보는 지어내지 마세요.

collect_info가 error를 반환하면 매장 평가를 지어내지 말고 사용자에게 정보 수집 실패를 알리세요.
"""

agent = LlmAgent(
    model="gemini-3.1-flash-lite",
    name="place_finder",
    description="지역 기반 장소 추천 에이전트",
    instruction=INSTRUCTION,
    tools=[collect_places, collect_info, read_file],
    output_schema=FeedOutput,
)

runner = InMemoryRunner(agent=agent, app_name=APP_NAME)

In [46]:
import uuid
from google.genai import types

async def run(query: str):
    session_id = str(uuid.uuid4())
    await runner.session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    content = types.Content(role="user", parts=[types.Part(text=query)])

    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.function_call:
                    print(f"[TOOL CALL] {part.function_call.name}({dict(part.function_call.args)})")
                elif part.function_response:
                    resp = part.function_response.response
                    # 파일 내용은 길어서 요약만 출력
                    if isinstance(resp, dict) and "content" in resp:
                        print(f"[TOOL RESULT] read_file → {len(str(resp['content']))}자")
                    else:
                        print(f"[TOOL RESULT] {resp}")
                elif part.text and part.text.strip():
                    print(f"[TEXT] {part.text.strip()}")

await run("마산에선 어머니 모시고 가기 좋은 축제 있어??")

[TOOL CALL] collect_places({'query': '마산 축제'})
[TOOL RESULT] {'status': 'retry', 'message': "'마산 축제' 검색 결과가 5개로 부족합니다. (1/3번째 시도) 핵심 지역명은 유지하고 수식어를 제거해 더 짧은 업종어로 바꿔 다시 호출하세요. 예: '조용히 작업하기 좋은 카페' → '카공카페' 또는 '스터디카페'"}
[TOOL CALL] collect_places({'query': '마산 관광지', 'category_code': 'AT4'})
[TOOL RESULT] {'status': 'ok', 'count': 40, 'source_breakdown': {'local': 40, 'blog_derived': 0}, 'file': 'tool_file\\1786596571208_collect_places.json'}
[TOOL CALL] collect_info({'user_query': '마산에서 어머니 모시고 가기 좋은 산책하기 좋은 관광지나 축제 장소'})
[429] '마산명물아구찜거리' (attempt 1/3)
[429] '돝섬' (attempt 1/3)
[429] '남파랑길 10코스' (attempt 1/3)
[429] '진북산업 진북황실온천' (attempt 1/3)
[429] '청량산' (attempt 1/3)
[429] '거락계곡' (attempt 1/3)
[429] '성호서7 가고파 꼬부랑길벽화마을' (attempt 1/3)
[429] '서원곡유원지' (attempt 1/3)
[429] '저도비치로드 1구간' (attempt 1/3)
[429] '팔의사 마빈온천' (attempt 1/3)
[429] '신마산통술거리' (attempt 1/3)
[429] '동서북17 오동동아구찜거리' (attempt 1/3)
[429] '마산어시장 장어구이거리' (attempt 1/3)
[429] '오동동문화의거리' (attempt 1/3)
[429] '용마산' (attempt 1/3)
[429] '